In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 02 · Delegation and token exchange

**Primer section:** §3.5 delegation mechanics (RFC 8693 token exchange, RFC 9449 DPoP,
RFC 8705 mTLS-bound tokens, Credential Access Boundaries) · §4.3 scoped credential per tool call.

The third idea in the primer: *least privilege per action, not per agent*. The credential used for
a tool call should have the narrowest audience, the narrowest scope, the shortest lifetime, and be
bound to the caller. Token exchange makes that cheap: the user's login token plus the agent's own
token go into the STS, and out comes a token whose `sub` is the user, whose `act` is the agent,
scoped to one resource. You will draw this on a whiteboard, so this notebook builds it claim by claim.

In [ ]:
from agentsec.logging_utils import quiet_logs

quiet_logs()

import json

import jwt  # display only: unverified peeks

from agentsec.identity import (
    AgentIdentity,
    AuthorityContext,
    AuthorityMode,
    BindingMismatch,
    BoundaryEvaluator,
    BoundaryRule,
    DPoP,
    ExpiredToken,
    InsufficientScope,
    InvalidAudience,
    LocalRuntimeCA,
    ReplayDetected,
    TokenIssuer,
    UserPrincipal,
    build_boundary,
    jwk_thumbprint,
    public_jwk,
)


def peek(token: str) -> dict:
    return jwt.decode(token, options={"verify_signature": False})

def short(spiffe: str) -> str:
    return spiffe.rsplit("/", 1)[-1]

ORG, PROJECT = "123456789012", "987654321098"
agent = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="support-agent", org_id=ORG)
sub_agent = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="refunds-subagent", org_id=ORG)
ana = UserPrincipal(subject="u-ana", email="ana@customer.example", tenant="acme")

ca = LocalRuntimeCA()
cert = ca.issue(agent)
issuer = TokenIssuer()          # the local STS / authorization server

FRONT_END = "https://app.acme.example"      # audience of the user's login token
TICKETS_MCP = "https://tickets.example/mcp" # canonical URI of a tool server (RFC 8707 resource)
PAYMENTS = "https://payments.example"

## 1. The inputs: a user token and an agent token

* The **subject token** is what the IdP gave the front-end after login (audience = the front-end).
  Its scopes are what the user consented to.
* The **actor token** is the agent's own certificate-bound token, addressed to the STS itself.

In [ ]:
user_token = issuer.mint(
    subject=ana.subject, audience=FRONT_END, scope="tickets:read tickets:write", extra={"email": ana.email}
)
actor_token = issuer.mint_agent_token(cert, audience=issuer.issuer)

print("subject token:", {k: peek(user_token)[k] for k in ("sub", "aud", "scope")})
print("actor token  :", {"sub": short(peek(actor_token)["sub"]), "aud": peek(actor_token)["aud"], "cnf": peek(actor_token)["cnf"]})

## 2. RFC 8693 exchange → `sub` = user, `act.sub` = agent

The STS verifies both tokens (the actor token must be presented with the agent's certificate —
binding is enforced at the STS too), then mints a **delegated** token for the requested audience.
Note what the response looks like: it is the standard token-exchange response shape.

In [ ]:
resp = issuer.exchange(
    subject_token=user_token, subject_token_audience=FRONT_END,
    actor_token=actor_token, actor_token_audience=issuer.issuer, presented_thumbprint=cert.thumbprint,
    audience=TICKETS_MCP, scope="tickets:read",
)
print("token response:", {k: (v[:24] + "…" if k == "access_token" else v) for k, v in resp.items()})

delegated = resp["access_token"]
claims = peek(delegated)
print(json.dumps({k: claims[k] for k in ("iss", "sub", "aud", "scope", "authority", "email", "act", "cnf")}, indent=2))

The resource server (the tickets MCP server) verifies the token **for its own audience** and, because
the token is still bound to the agent's certificate, requires the agent to present that certificate.
`Claims` exposes the delegation conveniently, and `AuthorityContext.from_claims` turns it into the
object policy and audit use.

In [ ]:
verified = issuer.verify(delegated, audience=TICKETS_MCP, presented_thumbprint=cert.thumbprint)
print("subject      :", verified.subject)
print("actor        :", short(verified.actor))
print("scopes       :", verified.scopes)
print("is_delegated :", verified.is_delegated)

ctx = AuthorityContext.from_claims(verified)
print("authority    :", ctx.audit_identities())
assert ctx.mode is AuthorityMode.DELEGATED and ctx.user.email == ana.email and ctx.agent == agent

## 3. Scope narrowing: a delegated token can never be broader than the user's consent

Ask for `tickets:read payments:refund`: the user never consented to `payments:refund`, so the
STS silently drops it. The intersection is the rule — the agent cannot escalate through the exchange.

In [ ]:
resp2 = issuer.exchange(
    subject_token=user_token, subject_token_audience=FRONT_END,
    actor_token=actor_token, actor_token_audience=issuer.issuer, presented_thumbprint=cert.thumbprint,
    audience=TICKETS_MCP, scope="tickets:read payments:refund",
)
print("requested : tickets:read payments:refund")
print("granted   :", resp2["scope"])
assert set(resp2["scope"].split()) == {"tickets:read"}

## 4. Nested chains: agent → sub-agent

When the support agent hands work to a refunds sub-agent it does **not** forward its token; it
exchanges again, with the sub-agent as the new actor. The previous `act` is nested (`act.act`), so
the whole chain is visible to the final resource server and to the audit log. Scopes only narrow.

In [ ]:
sub_agent_cert = ca.issue(sub_agent)
sub_actor_token = issuer.mint_agent_token(sub_agent_cert, audience=issuer.issuer)

resp3 = issuer.exchange(
    subject_token=delegated, subject_token_audience=TICKETS_MCP,
    actor_token=sub_actor_token, actor_token_audience=issuer.issuer, presented_thumbprint=sub_agent_cert.thumbprint,
    audience=PAYMENTS, scope="tickets:read",
)
hop2 = issuer.verify(resp3["access_token"], audience=PAYMENTS, presented_thumbprint=sub_agent_cert.thumbprint)
print("act claim  :", json.dumps(hop2.raw["act"], indent=2).replace("spiffe://agents.global.org-123456789012.system.id.goog/resources/aiplatform/projects/987654321098/locations/us-central1/reasoningEngines/", "…/"))
print("actor_chain:", [short(a) for a in hop2.actor_chain])
print("authority  :", AuthorityContext.from_claims(hop2).audit_identities())
assert hop2.actor_chain == [sub_agent.spiffe_id, agent.spiffe_id] and hop2.subject == ana.subject
assert hop2.cnf["x5t#S256"] == sub_agent_cert.thumbprint  # bound to the *current* actor's certificate

## 5. What the resource server checks, and the specific failures

Every verifier checks signature, issuer, expiry, **audience**, scopes and binding — and fails
with a distinct error so policy and logs can tell them apart.

In [ ]:
def outcome(label, fn):
    try:
        fn()
        print(f"{label:<42} accepted")
    except (InvalidAudience, InsufficientScope, ExpiredToken, BindingMismatch) as e:
        print(f"{label:<42} {type(e).__name__}: {e}")

outcome("right audience + cert", lambda: issuer.verify(delegated, audience=TICKETS_MCP, presented_thumbprint=cert.thumbprint))
outcome("same token sent to payments API", lambda: issuer.verify(delegated, audience=PAYMENTS, presented_thumbprint=cert.thumbprint))
outcome("server requires tickets:write", lambda: issuer.verify(delegated, audience=TICKETS_MCP, required_scopes={"tickets:write"}, presented_thumbprint=cert.thumbprint))
outcome("presented without the certificate", lambda: issuer.verify(delegated, audience=TICKETS_MCP))
expired = issuer.mint(subject="u-ana", audience=TICKETS_MCP, ttl=-30)
outcome("expired token", lambda: issuer.verify(expired, audience=TICKETS_MCP))

## 6. DPoP (RFC 9449): proof of possession per request

Where mTLS is not available end to end (through a gateway, from a public client), the client signs
a small JWT per request — `htm`, `htu`, `iat`, `jti`, and `ath` (hash of the access token) — with a
key it holds; the token carries `cnf.jkt`, the thumbprint of that key. The server verifies the proof,
checks `ath`, compares `jkt`, and remembers `jti` to refuse replays.

In [ ]:
key = DPoP.generate_key()
jwk = public_jwk(key)
dpop_token = issuer.mint_dpop_bound_token(subject="u-ana", audience=TICKETS_MCP, dpop_public_jwk=jwk, scope="tickets:read")
print("token cnf:", peek(dpop_token)["cnf"], "| jwk thumbprint:", jwk_thumbprint(jwk))

proof = DPoP.proof(key, method="POST", url=TICKETS_MCP, access_token=dpop_token)
print("proof header:", {k: v for k, v in jwt.get_unverified_header(proof).items() if k != "jwk"}, "| claims:", {k: v for k, v in peek(proof).items() if k in ("htm", "htu", "ath")})

seen_jti: set[str] = set()   # the server's replay cache
proof_claims = DPoP.verify(proof, method="POST", url=TICKETS_MCP, access_token=dpop_token, seen_jti=seen_jti)
verified_dpop = issuer.verify(dpop_token, audience=TICKETS_MCP, presented_jkt=proof_claims["jkt"])
print("first use  : accepted for", verified_dpop.subject)

for label, fn in [
    ("replayed proof (same jti)", lambda: DPoP.verify(proof, method="POST", url=TICKETS_MCP, access_token=dpop_token, seen_jti=seen_jti)),
    ("proof bound to a different token (ath)", lambda: DPoP.verify(proof, method="POST", url=TICKETS_MCP, access_token=dpop_token + "x")),
    ("proof for another URL (htu)", lambda: DPoP.verify(proof, method="POST", url="https://evil.example/mcp", access_token=dpop_token)),
    ("proof signed by the wrong key", lambda: issuer.verify(dpop_token, audience=TICKETS_MCP, presented_jkt=DPoP.verify(DPoP.proof(DPoP.generate_key(), method="POST", url=TICKETS_MCP, access_token=dpop_token), method="POST", url=TICKETS_MCP, access_token=dpop_token)["jkt"])),
]:
    try:
        fn()
        print(f"{label:<40} accepted (!)")
    except (ReplayDetected, BindingMismatch, Exception) as e:
        print(f"{label:<40} {type(e).__name__}: {e}")

## 7. Credential Access Boundaries: "exactly this prefix, for five minutes"

A CAB downscopes a Google access token to specific buckets/prefixes and permissions (Cloud Storage
only today). It is the pattern for handing a *tool call* a credential that cannot reach anything
else, whatever the source credential could do. `build_boundary` produces the JSON the STS accepts
(`google.auth.downscoped` consumes the same rules); `BoundaryEvaluator` emulates the check.

In [ ]:
boundary = build_boundary(
    BoundaryRule(bucket="acme-invoices", prefix="tenant-a/", roles=("roles/storage.objectViewer",)),
)
print(json.dumps(boundary.to_json(), indent=2))

ev = BoundaryEvaluator(boundary)
checks = [
    ("gs://acme-invoices/tenant-a/2026-01.pdf", "storage.objects.get"),
    ("gs://acme-invoices/tenant-b/2026-01.pdf", "storage.objects.get"),
    ("gs://acme-invoices/tenant-a/2026-01.pdf", "storage.objects.delete"),
    ("gs://other-bucket/tenant-a/x", "storage.objects.get"),
]
for uri, perm in checks:
    print(f"{ev.allows(uri, perm)!s:<6} {perm:<24} {uri}")
assert [ev.allows(u, p) for u, p in checks] == [True, False, False, False]

## On Google Cloud

* Google's STS implements the RFC 8693 grant (it is how Workload Identity Federation works); Agent
  Identity mints certificate-bound tokens; **Agent Gateway** enforces mTLS + DPoP ("double-bound").
* **Auth Manager** is the broker that hands the agent a user-delegated 3LO token per call (next
  notebook); Cloud Audit Logs then show both identities.
* CABs: `google.auth.downscoped.Credentials(source_credentials=ADC, credential_access_boundary=…)`
  — `agentsec.identity.downscope.downscoped_credentials(boundary)` wires it up on GCP.

**In one sentence:** "Delegation is a token exchange, never a token forward. The STS takes
the user's token and the agent's bound token and issues a token whose `sub` is the user and whose
`act` is the agent, scoped to one audience and to the intersection of scopes. Each hop exchanges
again so the chain is explicit, and sender-constraining — mTLS or DPoP — makes a stolen token
useless."